In [22]:
# Installations
!pip install openai
!pip install chromadb

In [ ]:
OPEN_AI_API_KEY="PROVIDE_YOUR_OPENAI_API_KEY"

In [19]:
import os
import chromadb
from openai import OpenAI
from chromadb.utils import embedding_functions

In [21]:
#Initialize openai embedding function
openai_ef = embedding_functions.OpenAIEmbeddingFunction(api_key=OPEN_AI_API_KEY, model_name="text-embedding-3-small")
openai_ef

In [51]:
# Initialize the Chroma client with persistence
chroma_client=chromadb.PersistentClient(path="chroma_persistent_storage")
collection_name="document_qa_collection"
collection = chroma_client.get_or_create_collection(name=collection_name, embedding_function=openai_ef)
collection

Collection(name=document_qa_collection)

In [25]:
#Initialize OpenAI client
openai_client=OpenAI(api_key=OPEN_AI_API_KEY)

In [27]:
response = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "What is human life expectancy in India?"},
    ]
)
print(response.choices[0].message.content)

The life expectancy in India is estimated to be around 69 years. Keep in mind that life expectancy can vary depending on factors such as gender, region, socioeconomic status, and access to healthcare.


In [28]:
# Function to load documents from a directory
def load_documents_from_directory(directory_path):
    print("==== Loading documents from directory ====")
    documents = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            with open(
                os.path.join(directory_path, filename), "r", encoding="utf-8"
            ) as file:
                documents.append({"id": filename, "text": file.read()})
    return documents

In [29]:
# Function to split text into chunks
def split_text(text, chunk_size=1000, chunk_overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

In [30]:
# Load documents from the directory
directory_path = "./data/news_articles"
documents = load_documents_from_directory(directory_path)
print(f"Loaded {len(documents)} documents")

==== Loading documents from directory ====
Loaded 21 documents


In [ ]:
# Split documents into chunks
chunked_documents = []
for doc in documents:
    chunks = split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({"id": f"{doc['id']}_chunk{i+1}", "text": chunk})

In [ ]:
# Function to generate embeddings using OpenAI API
def get_openai_embedding(text):
    response = openai_client.embeddings.create(input=text, model="text-embedding-3-small")
    embedding = response.data[0].embedding
    return embedding

# Generate embeddings for the document chunks
for doc in chunked_documents:
    doc["embedding"] = get_openai_embedding(doc["text"])
print(doc)

In [ ]:
# Upsert documents with embeddings into Chroma
for doc in chunked_documents:
    collection.upsert(
        ids=[doc["id"]], documents=[doc["text"]], embeddings=[doc["embedding"]]
    )

In [55]:
# Function to query documents
def query_documents(question, n_results=2):
    # query_embedding = get_openai_embedding(question)
    results = collection.query(query_texts=question, n_results=n_results)

    # Extract the relevant chunks
    relevant_chunks = [doc for sublist in results["documents"] for doc in sublist]
    print("==== Returning relevant chunks ====")
    return relevant_chunks
    # for idx, document in enumerate(results["documents"][0]):
    #     doc_id = results["ids"][0][idx]
    #     distance = results["distances"][0][idx]
    #     print(f"Found document chunk: {document} (ID: {doc_id}, Distance: {distance})")

In [57]:
# Function to generate a response from OpenAI
def generate_response(question, relevant_chunks):
    context = "\n\n".join(relevant_chunks)
    prompt = (
        "You are an assistant for question-answering tasks. Use the following pieces of "
        "retrieved context to answer the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the answer concise."
        "\n\nContext:\n" + context + "\n\nQuestion:\n" + question
    )

    response = openai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": question,
            },
        ],
    )

    answer = response.choices[0].message
    return answer

In [59]:
# Example query
# query_documents("tell me about AI replacing TV writers strike.")
# Example query and response generation
question = "tell me about databricks"
relevant_chunks = query_documents(question)
answer = generate_response(question, relevant_chunks)

print(answer)

==== Returning relevant chunks ====
ChatCompletionMessage(content="Databricks is a company that offers a unified data analytics platform that helps organizations harness the power of data. They recently acquired Okera, a company specializing in data governance and access control solutions. Databricks plans to integrate Okera's technology into its existing governance solution and expand their capabilities in managing data access policies at scale.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
